# OE4: comprobación reproducible en Google Colab

**Objetivo oficial (literal).** Medir y comparar el desempeño computacional del artefacto tecnológico desarrollado bajo diferentes condiciones de simulación, utilizando como indicadores el tiempo de convergencia, la latencia y el throughput.

**Qué comprueba este cuaderno.** Comprueba el objetivo en dos etapas ejecutadas en orden: etapa 1, recálculo de los indicadores por corrida (216 corridas válidas) desde cada petición registrada y encadenamiento con la tabla por celda; etapa 2, análisis pre-registrado (ANOVA, contrastes pareados con Holm, comprobaciones no paramétricas, estabilidad y sensibilidad) y exploratorio de interferencia desde los indicadores por corrida.

**Datos de entrada.** Registros por petición de las 216 corridas, `runs.jsonl`, `intentos.jsonl`, monitor de recursos de la sesión, corridas de OE2-B como referencia descriptiva y los scripts de análisis, con los resultados locales para comparar.

**Qué no hace.** No recalcula los tiempos medidos, que son los del equipo local con interferencia registrada. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Ejecute las celdas en orden en un entorno nuevo de Colab; la carga de datos pide subir los archivos indicados.

**Contenido (diez pasos).** 1 descripción; 2 datos de entrada; 3 verificación de archivos y estructura; 4 dependencias; 5 ejecución del análisis; 6 reconstrucción de los indicadores; 7 comparación con los resultados oficiales; 8 diferencias numéricas; 9 verificación de las salidas finales; 10 conclusión de reproducibilidad (calculada por la última celda).

In [ ]:
import sys, platform, os
print('Python', sys.version); print(platform.platform())
!pip -q install numpy scipy pandas statsmodels
import numpy, scipy, pandas, statsmodels; print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| pandas', pandas.__version__, '| statsmodels', statsmodels.__version__)
print('CPU', os.cpu_count())

In [ ]:
import zipfile, glob
PARTES = 4
NECESARIOS = ['oe4_colab_paquete.zip']
partes = sorted(glob.glob('oe4_http_p*.zip'))
faltan = [z for z in NECESARIOS if not os.path.exists(z)]
if len(partes) < PARTES or faltan:
    from google.colab import files
    print('Suba los archivos:', NECESARIOS, 'y las', PARTES, 'partes oe4_http_p1.zip ... oe4_http_p'+str(PARTES)+'.zip')
    files.upload()
    partes = sorted(glob.glob('oe4_http_p*.zip'))
assert len(partes) == PARTES, ('partes encontradas:', partes)
assert all(os.path.exists(z) for z in NECESARIOS), 'faltan paquetes de analisis'
for z in NECESARIOS + partes: zipfile.ZipFile(z).extractall('.')
print(len(partes), 'partes http extraidas + paquetes de analisis', NECESARIOS, '|', sorted(os.listdir('http')))

## Etapa 1: indicadores por corrida desde las peticiones registradas

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('http/hashes_paquete_http_oe4.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); n+=1
    if sha(os.path.join('http',nm))!=h: bad.append(nm)
print('archivos del paquete HTTP verificados contra el hash local:', n, '| distintos:', bad[:5]); assert not bad
base='http/oe4/campana_01'; ok=0; mal=[]
for l in open(base+'/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join(base,nm)
    if os.path.exists(p):
        ok+=1
        if sha(p)!=h: mal.append(nm)
print('oe4: archivos de la campaña presentes y verificados contra su SHA256SUMS.txt:', ok, '| distintos:', mal); assert not mal

In [ ]:
os.makedirs('http/out', exist_ok=True)
for k in ('oe4',):
    !python http/recomputo_http.py http/{k}/campana_01 http/out/recomputo_{k}.csv http/out/resumen_{k}.json > http/out/log_{k}.txt 2>&1
import json
print('Recálculo INDEPENDIENTE, desde cada petición registrada por el generador de carga, de los indicadores por corrida (12 campos) y comparación con lo guardado por el ejecutor:')
for k in ('oe4',):
    r=json.load(open(f'http/out/resumen_{k}.json'))
    mx=max(r['diferencia_abs_max_por_campo'].values()); mr=max(r['diferencia_rel_max_por_campo'].values())
    print(f"{k}: intentos recalculados {r['intentos_con_raw_recalculados']} | corridas válidas finales recalculadas {r['corridas_validas_finales_recalculadas']} | con los 12 campos idénticos {r['corridas_validas_finales_con_todos_los_campos_iguales']} | diferencia absoluta máxima {mx} | relativa máxima {mr}")

In [ ]:
import pandas as pd, math
def runs_de(k): return {json.loads(l)['label']: json.loads(l) for l in open(f'http/{k}/campana_01/runs.jsonl', encoding='utf8') if l.strip()}
# OE4: tabla por celda (media geométrica de P95 y de T_conv, media de throughput, peticiones y errores) desde los valores RECALCULADOS
rc=pd.read_csv('http/out/recomputo_oe4.csv'); rc=rc[rc['valida_final']==True].copy(); rr=runs_de('oe4')
for c,k in (('carga','load'),('N','n_particles'),('workers','workers')): rc[c]=rc['label'].map(lambda s: rr[s][k])
rc['lp']=rc['latency_p95_ms_recalculado'].map(math.log); rc['lt']=rc['t_conv_ms_mean_recalculado'].map(math.log)
g=rc.groupby(['carga','N','workers']).agg(n=('label','size'), lp=('lp','mean'), lt=('lt','mean'), thr=('throughput_rps_recalculado','mean'), req=('requests_total_recalculado','sum'), err=('errors_recalculado','sum')).reset_index()
g['p95_geo_r']=g['lp'].map(math.exp); g['tconv_geo_r']=g['lt'].map(math.exp)
e=pd.read_csv('http/oe4/esperado_celdas.csv')
m=g.merge(e,left_on=['carga','N','workers'],right_on=['carga','N','workers'])
d=lambda x,y: float((m[x]-m[y]).abs().max())
print('OE4 por celda (36 celdas, 6 bloques): celdas comparadas', len(m), '| dif abs máx | P95 geométrica:', d('p95_geo_r','p95_geo'), '| T_conv geométrica:', d('tconv_geo_r','tconv_geo'), '| throughput media:', d('thr','thr_media'), '| peticiones:', d('req','peticiones'), '| errores:', d('err','errores'))

## Etapa 2: análisis desde los indicadores por corrida

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('oe4/hashes_paquete_oe4.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join('oe4',nm); n+=1
    if sha(p)!=h: bad.append(nm)
print('archivos del paquete verificados contra el hash local:', n, '| distintos:', bad); assert not bad
bad2=[]; ok2=[]
for l in open('oe4/campana_01/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join('oe4/campana_01',nm)
    if os.path.exists(p):
        ok2.append(nm)
        if sha(p)!=h: bad2.append(nm)
print('contra el SHA256SUMS.txt de la campana, archivos presentes:', ok2, '| distintos:', bad2); assert not bad2

## Conteo de corridas

In [ ]:
import json, pandas as pd
R=[json.loads(l) for l in open('oe4/campana_01/runs.jsonl',encoding='utf8') if l.strip()]
r=pd.DataFrame(R); print('registros:', len(r), '| validos:', int((r.valid==True).sum()), '| no validos:', int((r.valid!=True).sum()))
print(r.groupby(['load','n_particles','workers']).block.nunique().value_counts().to_dict(), '(celdas por numero de bloques)')

## 1. Análisis PRE-REGISTRADO (`analisis_oe4.py`, mismos argumentos que en local: runs, salida, monitor y referencia OE2-B descriptiva)

In [ ]:
os.makedirs('oe4/salida_colab',exist_ok=True)
!python oe4/analisis_oe4.py oe4/campana_01/runs.jsonl oe4/salida_colab/preregistrado oe4/monitor.csv oe4/runs_oe2b_referencia.jsonl > oe4/salida_colab/log_preregistrado.txt 2>&1
print(open('oe4/salida_colab/log_preregistrado.txt').read()[-1500:])

## 2. Análisis EXPLORATORIO de interferencia (NO pre-registrado)

In [ ]:
!python oe4/exploratorio_interferencia_oe4.py oe4/campana_01/runs.jsonl oe4/salida_colab/preregistrado/resultados_oe4.json oe4/salida_colab/exploratorio_interferencia_oe4.json > oe4/salida_colab/log_exploratorio.txt 2>&1
print(open('oe4/salida_colab/log_exploratorio.txt').read()[-700:])

## 3. Comparación Colab vs local (diferencias absolutas y relativas)

In [ ]:
import json, math
import pandas as pd
def hoja(x, p=''):
    if isinstance(x, dict):
        for k,v in x.items(): yield from hoja(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i,v in enumerate(x): yield from hoja(v, p+'/'+str(i))
    else: yield p, x
def comparar_json(ruta_colab, ruta_local, etiqueta):
    A=dict(hoja(json.load(open(ruta_colab,encoding='utf8')))); B=dict(hoja(json.load(open(ruta_local,encoding='utf8'))))
    mismas = set(A)==set(B); mx=0.0; mxrel=0.0; peor=None; n_num=0; n_dist=0; n_nonum=0; dist=[]
    for k in A:
        if k not in B: continue
        va, vb = A[k], B[k]
        if isinstance(va,(int,float)) and isinstance(vb,(int,float)) and not isinstance(va,bool) and not isinstance(vb,bool):
            n_num+=1; dif=abs(va-vb); rel=dif/max(abs(vb),1e-300) if dif>0 else 0.0
            if dif>mx: mx=dif; peor=(k,va,vb)
            mxrel=max(mxrel, rel)
            if dif>0: n_dist+=1
            if dif>1e-9 and rel>1e-9: dist.append((k,va,vb,dif,rel))
        else:
            n_nonum+=1
            if va!=vb: dist.append((k,va,vb,None,None))
    print(f'[{etiqueta}] mismas claves: {mismas} | hojas: {len(A)} (numericas {n_num}, no numericas {n_nonum})')
    print(f'   diferencia absoluta maxima: {mx:.3e} | relativa maxima: {mxrel:.3e} | hojas numericas con diferencia > 0: {n_dist} | hojas con diferencia > 1e-9 (abs y rel): {len(dist)}')
    print('   peor hoja:', peor)
    for d in dist[:20]: print('   DIFERENCIA', d)
    return dict(etiqueta=etiqueta, mismas_claves=mismas, hojas=len(A), numericas=n_num, dif_abs_max=mx, dif_rel_max=mxrel, hojas_con_dif=n_dist, hojas_dif_gt_1e9=len(dist))
def comparar_csv(ruta_colab, ruta_local, etiqueta):
    a=pd.read_csv(ruta_colab); b=pd.read_csv(ruta_local)
    ok = list(a.columns)==list(b.columns) and a.shape==b.shape
    mx=0.0; mxrel=0.0; nd=0
    if ok:
        for c in a.columns:
            if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c]) and not pd.api.types.is_bool_dtype(a[c]) and not pd.api.types.is_bool_dtype(b[c]):
                dif=(a[c]-b[c]).abs().fillna(0); mx=max(mx,float(dif.max()))
                rel=(dif/b[c].abs().clip(lower=1e-300)).where(dif>0,0.0); mxrel=max(mxrel,float(rel.max())); nd+=int((dif>0).sum())
            else:
                nd+=int((a[c].astype(object).where(a[c].notna(),'<NA>').astype(str)!=b[c].astype(object).where(b[c].notna(),'<NA>').astype(str)).sum())
    print(f'[{etiqueta}] misma forma y columnas: {ok} {a.shape} | dif abs max: {mx:.3e} | rel max: {mxrel:.3e} | celdas distintas: {nd}')
    return dict(etiqueta=etiqueta, misma_forma=ok, filas=a.shape[0], dif_abs_max=mx, dif_rel_max=mxrel, celdas_distintas=nd)

In [ ]:
res=[]
res.append(comparar_json('oe4/salida_colab/preregistrado/resultados_oe4.json','oe4/esperado_local/resultados_oe4.json','OE4 pre-registrado: resultados_oe4.json'))
for f in ('celdas','contrastes','anova_p95','anova_thr','anova_tconv'):
    res.append(comparar_csv(f'oe4/salida_colab/preregistrado/{f}.csv',f'oe4/esperado_local/{f}.csv',f'OE4 pre-registrado: {f}.csv'))
res.append(comparar_json('oe4/salida_colab/exploratorio_interferencia_oe4.json','oe4/esperado_local/exploratorio_interferencia_oe4.json','OE4 EXPLORATORIO: interferencia'))
pd.DataFrame(res).to_csv('oe4/salida_colab/comparacion_colab_vs_local.csv',index=False)
print(pd.DataFrame(res)[['etiqueta','dif_abs_max','dif_rel_max']].to_string())

In [ ]:
import hashlib
print('SHA-256 de los 6 archivos de resultados (Colab == local):')
for f in ('anova_p95.csv','anova_thr.csv','anova_tconv.csv','celdas.csv','contrastes.csv','resultados_oe4.json'):
    print(f, sha('oe4/salida_colab/preregistrado/'+f)==sha('oe4/esperado_local/'+f))

## 4. Indicadores principales (Colab | local)

In [ ]:
a=json.load(open('oe4/salida_colab/preregistrado/resultados_oe4.json',encoding='utf8')); b=json.load(open('oe4/esperado_local/resultados_oe4.json',encoding='utf8'))
print('registros/validas/no validas/celdas con 6 bloques/interferidas:', [a[k] for k in ('n_registros','n_validas','n_no_validas','celdas_con_6_bloques','interferidas')], '|', [b[k] for k in ('n_registros','n_validas','n_no_validas','celdas_con_6_bloques','interferidas')])
print('sensibilidad sin interferidas:', a['sensibilidad_sin_interferidas'])
print('contrastes significativos (Holm < 0.05) por respuesta|familia:'); print(pd.DataFrame({'colab':{k:v['significativos_holm_005'] for k,v in a['contrastes_resumen'].items()},'local':{k:v['significativos_holm_005'] for k,v in b['contrastes_resumen'].items()}}).to_string())
print('eta2 parcial por respuesta (terminos principales):')
for rk in ('p95','thr','tconv'):
    ta={t['termino']:t for t in a['anova'][rk]['terminos']}; tb={t['termino']:t for t in b['anova'][rk]['terminos']}
    print(rk, {t:(round(ta[t]['eta2_parcial'],6), round(tb[t]['eta2_parcial'],6)) for t in ('C(load)','C(N)','C(workers)','C(load):C(workers)','C(block)')})
c=pd.read_csv('oe4/salida_colab/preregistrado/celdas.csv'); cl=pd.read_csv('oe4/esperado_local/celdas.csv')
print('celdas P95<2s (<=25 usuarios), Colab vs local:'); print(pd.concat([c[c.carga<=25].groupby(['workers','N']).p95_lt_2s.sum().rename('colab'), cl[cl.carga<=25].groupby(['workers','N']).p95_lt_2s.sum().rename('local')],axis=1).to_string())
print('celdas con throughput IC inferior >= 20 req/s:', int(c.thr_ge_20.sum()), int(cl.thr_ge_20.sum()))
print('estabilidad (Colab):', json.dumps({k:v for k,v in a['estabilidad'].items() if k in ('p95','thr','tconv')})[:600])

## Conclusión de la reproducción
Si la comparación muestra las mismas claves, diferencias nulas (o de coma flotante) y SHA-256 idénticos de los seis archivos de resultados, el análisis de OE4 **se reproduce** en Colab a partir de los datos originales. Esto **verifica el análisis**; **no convierte la campaña en un experimento controlado**: las 216 corridas siguen identificadas como interferidas.

## Conclusión de reproducibilidad

In [ ]:
import json, os
def _hojas(x, p=''):
    if isinstance(x, dict):
        for k, v in x.items(): yield from _hojas(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i, v in enumerate(x): yield from _hojas(v, p+'/'+str(i))
    else: yield p, x
def _verif(a, b):
    A = dict(_hojas(json.load(open(a, encoding='utf8')))); B = dict(_hojas(json.load(open(b, encoding='utf8'))))
    mx = 0.0; ng = 0; malas = []
    for k in A:
        if k in B and isinstance(A[k], (int, float)) and isinstance(B[k], (int, float)) and not isinstance(A[k], bool) and not isinstance(B[k], bool):
            d = abs(A[k]-B[k]); mx = max(mx, d)
            if d > 1e-9 and d/max(abs(B[k]), 1e-300) > 1e-9: ng += 1; malas.append((k, A[k], B[k]))
        elif k in B and A[k] != B[k]: ng += 1; malas.append((k, A[k], B[k]))
    return set(A) == set(B), len(A), mx, ng, malas
CHECKS = [('OE4 pre-registrado resultados_oe4.json', 'oe4/salida_colab/preregistrado/resultados_oe4.json', 'oe4/esperado_local/resultados_oe4.json'), ('OE4 exploratorio interferencia', 'oe4/salida_colab/exploratorio_interferencia_oe4.json', 'oe4/esperado_local/exploratorio_interferencia_oe4.json')]
HTTP = ['oe4']
print('CONCLUSION DE REPRODUCIBILIDAD')
todo_ok = True
dep_version = False
for etiqueta, a, b in CHECKS:
    mismas, n, mx, ng, malas = _verif(a, b)
    solo_wilcoxon = ng > 0 and all(k.endswith('/p_wilcoxon') for k, _, _ in malas)
    ok = mismas and ng == 0; todo_ok &= ok or (mismas and solo_wilcoxon); dep_version |= (not ok) and mismas and solo_wilcoxon
    veredicto = 'COINCIDE' if ok else ('COINCIDE salvo p_wilcoxon (aproximación normal que depende de la versión de scipy)' if mismas and solo_wilcoxon else 'NO COINCIDE')
    print(f'  {etiqueta}: misma estructura {mismas} | hojas {n} | diferencia absoluta máxima {mx:.3e} | hojas con diferencia > 1e-9: {ng} -> {veredicto}')
    for k, x, y in malas[:10]: print(f'     hoja distinta: {k} | Colab {x!r} | local {y!r}')
for k in HTTP:
    r = json.load(open(f'http/out/resumen_{k}.json'))
    ok = r['corridas_validas_finales_recalculadas'] == r['corridas_validas_finales_con_todos_los_campos_iguales'] and max(r['diferencia_abs_max_por_campo'].values()) == 0
    todo_ok &= ok
    print(f"  {k}: indicadores por corrida recalculados desde las peticiones: {r['corridas_validas_finales_recalculadas']} corridas válidas, {r['corridas_validas_finales_con_todos_los_campos_iguales']} idénticas -> {'COINCIDE' if ok else 'NO COINCIDE'}")
if todo_ok and dep_version: print('Resultado: el análisis se reproduce con los datos oficiales; todas las hojas coinciden dentro de 1e-9 salvo los p de Wilcoxon listados arriba, que dependen de la versión de scipy instalada (scipy', __import__('scipy').__version__, 'en este entorno). Las demás cantidades, incluido el ajuste de Holm calculado con p_t, coinciden.')
else: print('Resultado:', 'el análisis se reproduce con los datos oficiales y coincide con el resultado local dentro de la tolerancia de 1e-9 (las diferencias de coma flotante se informan arriba).' if todo_ok else 'HAY DIFERENCIAS: revise las líneas marcadas como NO COINCIDE antes de concluir.')
print('Alcance: se vuelve a ejecutar el análisis estadístico sobre los datos de entrada; no se repiten las campañas ni se regeneran los ciclos o las corridas, y la reproducción no implica que las condiciones de adquisición fueran controladas.')